# Export FAISS Index for Demo App
This notebook runs the exact same logic as your original pipeline to load the `bitext/Bitext-customer-support-llm-chatbot-training-dataset`, split out the 250 test evaluation rows, and embed the remaining knowledge base using `sentence-transformers/all-MiniLM-L6-v2`. Finally, it exports the `data` artifacts needed for your Streamlit demo app.

In [ ]:
!pip install datasets sentence-transformers faiss-cpu pandas pyarrow

In [ ]:
import random
from collections import defaultdict
from datasets import load_dataset
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
import hashlib
from datetime import datetime
import os
from google.colab import files

In [ ]:
# Configuration exactly as in your notebook
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
DATASET_SPLIT = "train"

EVAL_TOTAL_SIZE     = 500
SET_SIZE            = 250
LORA_TRAIN_FRACTION = 0.05
STRATIFY_SEED       = 17
EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

print('Loading dataset...')
_full_list = list(load_dataset(DATASET_NAME, split=DATASET_SPLIT))

# Preserve exactly the same splitting logic
random.seed(42)
_eval_idx = list(random.sample(range(len(_full_list)), EVAL_TOTAL_SIZE))
random.Random(43).shuffle(_eval_idx)

_set_a_idx = set(_eval_idx[:SET_SIZE])
set_a_rows = [row for i, row in enumerate(_full_list) if i in _set_a_idx]

_remaining = [(i, row) for i, row in enumerate(_full_list) if i not in _set_a_idx]
_by_intent = defaultdict(list)
for i, row in _remaining:
    _by_intent[row.get('intent', 'unknown')].append((i, row))

_rng = random.Random(STRATIFY_SEED)
index_pairs = []
for intent, rows in _by_intent.items():
    _rng.shuffle(rows)
    n_train = round(len(rows) * LORA_TRAIN_FRACTION)
    index_pairs.extend(rows[n_train:])

index_rows = [row for _, row in index_pairs]
print(f'Held-out set A (Evaluation): {len(set_a_rows)} rows')
print(f'Final FAISS index size: {len(index_rows)} rows')

In [ ]:
# Format the text chunks perfectly matching the Streamlit app
all_chunks = []
for row in index_rows:
    chunk = (
        f"[Category: {row.get('category','')}] [Intent: {row.get('intent','')}]\n"
        f"Q: {row.get('instruction','')}\nA: {row.get('response','')}"
    )
    all_chunks.append(chunk)

print(f'Embedding {len(all_chunks)} chunks using {EMBEDDING_MODEL} (this takes ~2-3 minutes on Colab GPU)...')
embed_model = SentenceTransformer(EMBEDDING_MODEL)

all_vecs = []
# Process in batches of 256 for optimal GPU memory
for i in range(0, len(all_chunks), 256):
    vecs = embed_model.encode(
        all_chunks[i:i+256], normalize_embeddings=True, show_progress_bar=False)
    all_vecs.append(vecs)
    if (i // 256) % 5 == 0:
        print(f'  ... {min(i+256, len(all_chunks))}/{len(all_chunks)} embedded')

embeddings = np.vstack(all_vecs).astype(np.float32)

# Build the FAISS index
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print(f'FAISS index ready: {faiss_index.ntotal} vectors')

In [ ]:
# Export the 4 required files for your demo_app/data folder
out_dir = "data"
os.makedirs(out_dir, exist_ok=True)

# 1. FAISS Index
faiss.write_index(faiss_index, os.path.join(out_dir, "faiss.index"))

# 2. Excluded set A IDs (Leakage prevention check in your demo app)
def get_id(row):
    return hashlib.md5(json.dumps(row, sort_keys=True).encode('utf-8')).hexdigest()

set_a_ids = [get_id(row) for row in set_a_rows]
with open(os.path.join(out_dir, "set_a_ids.json"), "w") as f:
    json.dump(set_a_ids, f)

# 3. Parquet mapping of chunk IDs to texts
chunks_data = []
for i, row in enumerate(index_rows):
    chunks_data.append({
        "chunk_id": get_id(row),
        "text": all_chunks[i],
        "intent": row.get("intent", ""),
        "category": row.get("category", "")
    })

df = pd.DataFrame(chunks_data)
df.to_parquet(os.path.join(out_dir, "chunks.parquet"))

# 4. Meta configuration file
meta = {
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": faiss_index.d,
    "normalize_embeddings": True,
    "faiss_index_type": "IndexFlatIP",
    "n_chunks": faiss_index.ntotal,
    "dataset_name": DATASET_NAME,
    "export_date": datetime.now().isoformat()
}
with open(os.path.join(out_dir, "meta.json"), "w") as f:
    json.dump(meta, f)

print(f"Successfully generated all data files in {out_dir}")

In [ ]:
# Zip the folder and trigger a download to your local machine
!zip -r data.zip data/
print("\nTriggering download of data.zip...")
print("Please extract this and place the 4 files inside your local demo_app/data/ folder.")
files.download("data.zip")